In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("../data/UNSW_NB15_training-set.csv")
test  = pd.read_csv("../data/UNSW_NB15_testing-set.csv")

print("Original train shape:", train.shape)
print("Original test shape :", test.shape)

Original train shape: (175341, 45)
Original test shape : (82332, 45)


In [2]:
# 1) Drop leakage / non-predictive columns
# - id: identifier only
# - attack_cat: not used for binary classification; can leak label information
drop_cols = ["id", "attack_cat"]

train = train.drop(columns=drop_cols)
test  = test.drop(columns=drop_cols)

# 2) Separate features and target
y_train = train["label"].astype(int)
X_train = train.drop(columns=["label"])

y_test = test["label"].astype(int)
X_test = test.drop(columns=["label"])

print("Feature-only train shape:", X_train.shape)
print("Feature-only test shape :", X_test.shape)

Feature-only train shape: (175341, 42)
Feature-only test shape : (82332, 42)


In [3]:
# 3) Missing-value handling
# UNSW-NB15 is effectively clean, but this makes the pipeline safe and reusable.

cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

print("Categorical columns:", cat_cols)
print("Number of numeric columns:", len(num_cols))

print("\nMissing values before filling (train):")
print(X_train.isna().sum().sort_values(ascending=False).head(10))

for c in num_cols:
    median_value = X_train[c].median()
    X_train[c] = X_train[c].fillna(median_value)
    X_test[c] = X_test[c].fillna(median_value)

for c in cat_cols:
    X_train[c] = X_train[c].fillna("unknown")
    X_test[c] = X_test[c].fillna("unknown")

print("\nMissing values after filling (train):")
print(X_train.isna().sum().sort_values(ascending=False).head(10))

Categorical columns: ['proto', 'service', 'state']
Number of numeric columns: 39

Missing values before filling (train):
dur        0
proto      0
service    0
state      0
spkts      0
dpkts      0
sbytes     0
dbytes     0
rate       0
sttl       0
dtype: int64

Missing values after filling (train):
dur        0
proto      0
service    0
state      0
spkts      0
dpkts      0
sbytes     0
dbytes     0
rate       0
sttl       0
dtype: int64


In [4]:
# 5) One-hot encode categorical columns
X_train_enc = pd.get_dummies(X_train, drop_first=False)
X_test_enc  = pd.get_dummies(X_test, drop_first=False)

# 6) Align train/test to identical columns
X_train_enc, X_test_enc = X_train_enc.align(
    X_test_enc, join="outer", axis=1, fill_value=0
)

print("Encoded train shape:", X_train_enc.shape)
print("Encoded test shape :", X_test_enc.shape)

Encoded train shape: (175341, 196)
Encoded test shape : (82332, 196)


In [5]:
# 7) Save unscaled data for CatBoost / tree baselines
X_train_enc.to_csv("../data/X_train_enc.csv", index=False)
X_test_enc.to_csv("../data/X_test_enc.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

print("Saved unscaled files:")
print("X_train_enc.csv")
print("X_test_enc.csv")
print("y_train.csv")
print("y_test.csv")

Saved unscaled files:
X_train_enc.csv
X_test_enc.csv
y_train.csv
y_test.csv


In [6]:
# 8) Normalization / scaling
# Required for Neural Networks / FL / DP, not required for CatBoost.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc.values)
X_test_scaled = scaler.transform(X_test_enc.values)

np.save("../data/X_train_scaled.npy", X_train_scaled)
np.save("../data/X_test_scaled.npy", X_test_scaled)

print("Saved scaled arrays:")
print(" X_train_scaled.npy")
print(" X_test_scaled.npy")
print("\nScaled train shape:", X_train_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)

Saved scaled arrays:
 X_train_scaled.npy
 X_test_scaled.npy

Scaled train shape: (175341, 196)
Scaled test shape : (82332, 196)
